# CMU auto-graded notebook

Before you turn these assignments in, make sure everything runs as expected. First, **restart the kernel** (in the menubar, select Kernel$\rightarrow$Restart) and then **run all cells** (in the menubar, select Cell$\rightarrow$Run All).

Make sure you fill in any place that says `YOUR CODE HERE`, `<FILL IN>`, or "YOUR ANSWER HERE."


---


# CMU Machine Learning with Large Datasets

## Homework 1 - Coding 2: Streaming Naive Bayes


In [0]:
# Who did you collaborate with on this assignment?
# if no one, collaborators should contain an empty string,
# else list your collaborators below

# collaborators = [""]
# YOUR CODE HERE
raise NotImplementedError()

In [0]:
try:
    collaborators
except:
    raise AssertionError("you did not list your collaborators, if any")

### (1a) Environment Setup

Run the following cell to establish the runtime environment.


In [0]:
# Ignore this cell if the environment already has the packages

%pip install nose numpy

In [0]:
# imports that will be used in the notebook -- shouldn't need to import any other libraries

import math
import os
import re
from collections import Counter

import numpy as np
from nose.tools import assert_equal

### (1b) Data Preparation


We use the Reuters Corpus (RCV1), which is a set of news stories split into a hierarchy of categories. There are three file sets. The two data sets with "small" in the name contain smaller subsamples of the full data set. They are provided for debugging and local tests. The final classification task should use the "full" one. Each data set is split into a train and test set, as indicated by the file suffix:

- RCV1.full_train.txt
- RCV1.full_test.txt
- RCV1.small_train.txt
- RCV1.small_test.txt
- RCV1.very_small_train.txt
- RCV1.very_small_test.txt

Download the data using the link provided in the handout, and fill in the corresponding variables with their file paths in the following cell.


In [0]:
# Uncomment this cell if running on Databricks

from urllib.parse import urlparse
from pyspark import SparkFiles

def get_file(url):
    parsed_url = urlparse(url)
    suffix = parsed_url.path.split('/')[-1]
    sc.addFile(url)
    path = SparkFiles.get(suffix)

    return path

FULL_TRAIN = get_file("https://cmu.box.com/shared/static/g37w2q552qaik60yubrd3aq95szgi7pc")
FULL_TEST = get_file("https://cmu.box.com/shared/static/305174svzxe3lqm0ps76fvaenhr37inj")
SMALL_TRAIN = get_file("https://cmu.box.com/shared/static/f1x91jode5ywofu5b6cm9tb6ynx3yoft")
SMALL_TEST = get_file("https://cmu.box.com/shared/static/0l8y7x3gtix2sjfb1ewzq10c8ayaokdd")
VERY_SMALL_TRAIN = get_file("https://cmu.box.com/shared/static/xiumqmfdge5muxhs9v2w7ma97hurqtqy")
VERY_SMALL_TEST = get_file("https://cmu.box.com/shared/static/np2s7tn5wvwuawad2vddygs76tbwt1f5")

In [0]:
# TODO: Replace <FILL IN> with appropriate file paths
FULL_TRAIN = <FILL IN>
FULL_TEST = <FILL IN>
SMALL_TRAIN = <FILL IN>
SMALL_TEST = <FILL IN>
VERY_SMALL_TRAIN = <FILL IN>
VERY_SMALL_TEST = <FILL IN>

There are multiple class labels per document, meaning that there is more than one correct answer to the question "What kind of news article is this?"

For this assignment, we will ignore all class labels except for those ending in CAT and just be classifying into the top-level nodes of the hierarchy:

- CCAT: Corporate/Industrial
- ECAT: Economics
- GCAT: Government/Social
- MCAT: Markets

DO NOT change the following cell and just run it to set up the CAT labels


In [0]:
CAT_LABELS = ['CCAT', 'ECAT', 'GCAT', 'MCAT']

The data format is one document per line, with the class label(s) first (comma separated), a tab character, and then the document text.

Run the following cell to take a glance at the first document of the small training data set.


In [0]:
with open(SMALL_TRAIN, 'r') as f:
    print(f.readline())

C13,C21,CCAT	 The German government on Thursday awarded the first round of licences for basic public telephone services, opening the door for full competition to monopoly Deutsche Telekom. Kicking off the final stage of preparations ahead of liberalisation of European telecoms on January 1, 1998, Germany awarded licences for basic phone services to Vebacom GmbH, Britain's Colt Telecom Plc and NetCologne, a local operator.   The so-called &quot;class four&quot; licences cover basic voice telephone services for the public. That is the only service still under monopoly protection in Germany, where the telecoms market is expected to be worth more than 100 billion marks by the year 2000. In addition to the class 4 licences, the government awarded &quot;class 3&quot; licences to DBKom GmbH, the joint venture of a Mannesmann-led group and the German railway Deutsche Bahn AG, as well as to local operator VEW Telnet. Under a law passed this year, the class 3 licence allow owners of telecoms net

### (1c) Data Processing


To count the words, we need to tokenize the document text. In real-world practice, this may involve multiple steps, such as removing stop words and converting text to lowercase, which you learned in HW1: Entity Resolution. For this Naive Bayes part, we simplify the process by splitting only on words.

Run the following cell to define the `tokenization(doc)` function.


In [0]:
# DO NOT change this function
def tokenizeDoc(doc: str) -> list[str]:
    """
    Convert input document text into tokenization features
    Args:
        doc: document text
    Returns:
        list: a list of tokens
    """
    return re.findall('\\w+', doc)

As the handout instructs, streaming Naive Bayes loads one-line document at a time, use that document to update the classifier statistics, and then discard the document. After loading a line, we need a function to parse the line for classifier training.

Implement `parseDatafileLine(datafileLine)` that takes a line (labels + document text) and return a list of labels and a list of tokens. You need to use the `tokenizeDoc(doc)` function defined above.


In [0]:
def parseDatafileLine(datafileLine: str):
    """
    Parse a line of the data file to separate labels and document tokens
    Args:
        datafileLine: input string that is a line from the data file
    Returns:
        labels (list), tokens (list)
    """
    # TODO: YOUR CODE HERE
    labels_text = datafileLine.split('\t')
    labels = labels_text[0].split(',')
    text = labels_text[1]
    tokens = tokenizeDoc(text)
    return labels, tokens

In [0]:
"""Test parseDatafileLine(datafileLine)"""

line_sample1 = "C15,C151,CCAT\tMcDonald's Corp said Thursday it raised its quarterly dividend 10 percent, to $0.0825 a share from $0.075."
line_sample2 = "C15,C151,CCAT\tSix months to Sept 30, 1996       (in million rupees unless stated)"

assert_equal(parseDatafileLine(line_sample1),
             (['C15', 'C151', 'CCAT'],
              ['McDonald', 's', 'Corp', 'said', 'Thursday', 'it', 'raised', 'its',
               'quarterly', 'dividend', '10', 'percent', 'to', '0', '0825', 'a',
               'share', 'from', '0', '075']))

assert_equal(parseDatafileLine(line_sample2),
             (['C15', 'C151', 'CCAT'],
              ['Six', 'months', 'to', 'Sept', '30', '1996', 'in', 'million',
               'rupees', 'unless', 'stated']))

### 1(d) Training


Now, we are good for training. We will use a dictionary as the model to store the count statistics.

As the handout instructs, the model contains five parts:

- `y`: $(Y=y)$ for each label y the number of training instances of that class
- `ys`: $(Y=*)$ here $*$ means anything, so this is just the total number of training instances
- `y_w`: $(Y=y,W=w)$ number of times token w appears in a document with label y
- `y_ws`: $(Y=y,W=*)$ total number of tokens for documents with label y
- `vocabulary`: track the vocabulary for Laplace smoothing

Recall that Laplace smoothing formula is

$$p(W=w_i|Y=y)=\frac{count(Y=y,W=w_i) + \alpha}{count(Y=y,W=*)+ \alpha|V|}$$
where $|V|$ is the vocabulary.

Implement `nbmodel()` by figuring out the proper variable types for each part and filling in the blank below.


In [0]:
def nbmodel() -> dict:
    """
    Returns:
        dict storing the required five parts
    """
    # TODO: Replace <FILL IN> with appropriate code
    return {
        # 'y': {}, #event counters
        # 'ys': 0,
        # 'y_w': {},
        # 'y_ws': {},
        # 'vocabulary': set()
        'y': Counter(), #event counters
        'ys': 0,
        'y_w': Counter(),
        'y_ws': Counter(),
        'vocabulary': set()
    }

Implement `trainNB(trainfile, model)` that loads one document at a time and uses that document to update the required statistics for the Naive Bayes classifier.

Hint:

1. We only use those lables ending in CAT, which defined in `CAT_LABELS` earlier. Therefore, you need to figure out a way to skip other labels.
2. There are some documents with more than one CAT label. Treat those documents as multiple data instances (that is, add to the counters for
   all labels ending in CAT). For instance, if one line contains CCAT and ECAT, use the same document text twice.
3. It is not necessary to return the model here if you use proper types. Think about Python's mutable vs immutable types.


In [0]:
# def trainNB(trainfile: str, model: dict):
#     """
#     Update the model in a streaming fashion.
#     Args:
#         trainfile: file path of the training data
#         model: dict of the Naive Bayes classifier model
#     """
#     with open(trainfile, 'r', encoding='utf-8', errors='replace') as f:
#     #with open(trainfile, 'r') as f:
#         # Please note here we use f for iteration directly instead of f.readlines().
#         # Think about the reasons from the perspective of streaming
#         for line in f:
#             labels, tokens = parseDatafileLine(line)

#             relevant_labels = []
#             for label in labels:
#                 if label in CAT_LABELS:
#                     relevant_labels.append(label)
#             if not relevant_labels: 
#                 continue

#             model['ys'] += len(relevant_labels)

#             for label in relevant_labels:
#                 if label not in model['y']:
#                     model['y'][label] = 0
#                 if label not in model['y_w']:
#                     model['y_w'][label] = {}
#                 if label not in model['y_ws']:
#                     model['y_ws'][label] = 0
#                 model['y'][label] += 1
#                 for token in tokens:
#                     if token not in model['y_w'][label]:
#                         model['y_w'][label][token] = 0
#                     model['y_w'][label][token] += 1
#                     model['y_ws'][label] += 1
#                     model['vocabulary'].add(token)



In [0]:

def trainNB(trainfile: str, model: dict):
    """
    Update the model in a streaming fashion.
    Args:
        trainfile: file path of the training data
        model: dict of the Naive Bayes classifier model
    """
    with open(trainfile, 'r', encoding='utf-8', errors='replace') as f:
        for line in f:
            labels, tokens = parseDatafileLine(line)

            relevant_labels = [label for label in labels if label in CAT_LABELS]
            if not relevant_labels: 
                continue

            model['ys'] += len(relevant_labels)

            for label in relevant_labels:
                model['y'][label] += 1

                if label not in model['y_w']:
                    model['y_w'][label] = Counter()

                for token in tokens:
                    model['y_w'][label][token] += 1  

                model['y_ws'][label] += len(tokens)

                model['vocabulary'].update(tokens)


### 1(e) Test


For each line of documents, your classification code should get the best class $y$ and its log probabilities:

$$ln(p(Y=y))+\sum_{w_i} ln(p(W=w_i|Y=y))$$

Notice that we use the natural logarithm here.

Implement `testNB(testfile, model)` that uses the trained model to classify the test data and return a list of best classes, a list of max log probabilities (**rounding it to 4 decimal places**), and overall accuracy (**rounding it to 4 decimal places**). Please explicitly return in this specified order.


In [0]:
def testNB(testfile, model):
    """
    Implement Naive Bayes classification
    Args:
        testfile: file path of the test data
        model: dict of the Naive Bayes classifier model
    Returns:
        best_classes, log_probabilities, accuracy
    """
    best_classes = []
    log_probabilities = []
    
    correct = 0
    total = 0
    with open(testfile, 'r', encoding='utf-8', errors='replace') as f:
    #with open(testfile, 'r') as f:
        for line in f.readlines():
            labels, tokens = parseDatafileLine(line)

            log_prob = {}
            for y in model['y']:
                log_prob[y] = math.log(model['y'][y] / model['ys'])
                for token in tokens:
                    words = model['y_w'][y][token] + 1
                    total_count = model['y_ws'][y] + len(model['vocabulary'])
                    log_prob[y] += math.log( words / total_count)
            best_class = max(log_prob, key=log_prob.get)
            best_classes.append(best_class)
            log_probabilities.append(round(log_prob[best_class],4))

            if best_class in labels:
                correct += 1
            total += 1
    accuracy = correct / total
    return best_classes, log_probabilities, accuracy, correct, total

In [0]:
"""DO NOT change this this cell and just run it to use the very small dataset to test your implementations"""

very_small_model = nbmodel()
trainNB(VERY_SMALL_TRAIN, very_small_model)
best_classes, log_probabilities, accuracy = testNB(VERY_SMALL_TEST, very_small_model)

assert_equal(best_classes,
             ['MCAT', 'ECAT', 'CCAT', 'ECAT', 'CCAT', 'CCAT', 'ECAT', 'CCAT'])
assert_equal(log_probabilities,
             [-9893.7804, -3912.8180, -1121.5992, -1610.1660,
              -701.3466, -1453.3430, -2218.3302, -2285.0698])
assert_equal(accuracy, 0.8750)

### 1(f) Full Classification and Deliverable

We are almost there! Let's wrap up this assignment.

Implement your training and test functions on the full dataset to get the full classification results. Please note that you need to define a new model different from the `very_small_model` we have already tested.

Write the full classification results to a file `full_result.txt` (please explicitly use this name). The output format should have one test result per line, and each line should have the format:

$$\text{[Label1, Label2, ...]<tab>Best class<tab>Log prob}$$

where **[Label1, Label2, ...]** are the true labels of the test instance, **Best class** is the class with the maximum log probability, and the last field is the log probability.

The last line of the file should include the accuracy.

Use the following cell to write your code.

Submit both this notebook and `full_result.txt` to Gradescope.


In [0]:
full_model = nbmodel()
trainNB(FULL_TRAIN, full_model) 
best_classes, log_probabilities, accuracy, correct, total = testNB(FULL_TEST, full_model)

with open("full_result.txt", 'w') as f:
    with open(FULL_TEST, 'r') as test_f:
        for i, line in enumerate(test_f.readlines()):
            labels, _ = parseDatafileLine(line)
            
            result_line = f"{labels}\t{best_classes[i]}\t{log_probabilities[i]:.4f}\n"
            f.write(result_line)
    
    f.write(f"Accuracy: {correct}/{total}={accuracy:.4f}\n")


---------------------------------------------------------------------------
FileNotFoundError                         Traceback (most recent call last)
File <command-470243714821300>, line 5
      2 trainNB(FULL_TRAIN, full_model) 
      3 best_classes, log_probabilities, accuracy = testNB(FULL_TEST, full_model)
----> 5 with open("full_result.txt", 'w') as f:
      6     with open(FULL_TEST, 'r') as test_f:
      7         for i, line in enumerate(test_f.readlines()):

FileNotFoundError: [Errno 2] No such file or directory

In [0]:
# * Here is an expected output of very_small_test dataset for your reference
# * You need to write one using the FULL dataset

# import chardet
# with open(FULL_TRAIN, 'rb') as f:
#     raw_data = f.read(100000) 
#     result = chardet.detect(raw_data)
# print(result)
# import os

# print("Current working directory:", os.getcwd())
# print("Files in this directory:", os.listdir())


full_model = nbmodel()
trainNB(FULL_TRAIN, full_model)
best_classes, log_probabilities, accuracy = testNB(FULL_TEST, full_model)
print(FULL_TEST)  # Check if it exists

'''
['C24', 'CCAT', 'M14', 'MCAT']\tMCAT\t-9893.7804
['E51', 'E512', 'ECAT', 'GCAT', 'GDIP']\tECAT\t-3912.8180
['C15', 'C152', 'C18', 'C181', 'CCAT']\tCCAT\t-1121.5992
['GCAT']\tECAT\t-1610.1660
['C13', 'CCAT', 'GCAT', 'GHEA']\tCCAT\t-701.3466
['C13', 'CCAT', 'M11', 'MCAT']\tCCAT\t-1453.3430
['C11', 'C13', 'CCAT', 'E12', 'ECAT', 'M13', 'M132', 'MCAT']\tECAT\t-2218.3302
['C31', 'CCAT']\tCCAT\t-2285.0698
Accuracy: 7/8=0.8750
'''

/local_disk0/spark-661b53e6-5993-458b-b29e-c3762805acdb/userFiles-da83daa9-64c4-415c-8e3f-38a931318104/305174svzxe3lqm0ps76fvaenhr37inj


"\n['C24', 'CCAT', 'M14', 'MCAT']\tMCAT\t-9893.7804\n['E51', 'E512', 'ECAT', 'GCAT', 'GDIP']\tECAT\t-3912.8180\n['C15', 'C152', 'C18', 'C181', 'CCAT']\tCCAT\t-1121.5992\n['GCAT']\tECAT\t-1610.1660\n['C13', 'CCAT', 'GCAT', 'GHEA']\tCCAT\t-701.3466\n['C13', 'CCAT', 'M11', 'MCAT']\tCCAT\t-1453.3430\n['C11', 'C13', 'CCAT', 'E12', 'ECAT', 'M13', 'M132', 'MCAT']\tECAT\t-2218.3302\n['C31', 'CCAT']\tCCAT\t-2285.0698\nAccuracy: 7/8=0.8750\n"

In [0]:
len(log_probabilities)

80435

In [0]:
labelsFINAL = []
with open(FULL_TEST, 'r', encoding='utf-8', errors='replace') as f:
#with open(testfile, 'r') as f:
    for line in f.readlines():
        labels, tokens = parseDatafileLine(line)
        labelsFINAL.append(labels)


In [0]:
formatted_labels = [str(lst) for lst in labelsFINAL]


In [0]:
finals = ''
for i in range(len(formatted_labels)):
    finals += formatted_labels[i]
    finals += "\t"
    finals += best_classes[i]
    finals += "\t"
    finals += str(log_probabilities[i])
    finals += "\n"


In [0]:
workspace_path = "/Workspace/Users/nickachermak@gmail.com/full_results.txt"  # Replace with your
dbutils.fs.put(workspace_path, finals, overwrite=True)

Wrote 3320626 bytes.


True

In [0]:
import os
print(os.getcwd())
dbutils.fs.ls("/Workspace/Users/nickachermak@gmail.com")




/Workspace/Users/nickachermak@gmail.com


[FileInfo(path='dbfs:/Workspace/Users/nickachermak@gmail.com/full_results.txt', name='full_results.txt', size=3320626, modificationTime=1738213410000),
 FileInfo(path='dbfs:/Workspace/Users/nickachermak@gmail.com/my_output.txt', name='my_output.txt', size=3320626, modificationTime=1738212530000)]

In [0]:
dbutils.fs.cp("dbfs:/Workspace/Users/nickachermak@gmail.com/full_results.txt", "file:/tmp/full_results.txt")



True